# Relevia research: Colab runner

This notebook only **calls the scripts**. All the logic lives in `src/` and `configs/`, so every run is reproducible.

Runtime → Change runtime type → **GPU** (T4 is fine for 1–3B models, L4 is faster).

## 1. Get the project
Upload the `relevia-research` folder to Google Drive (under `MyDrive/`), then mount it. Adapters and results are saved straight to Drive, so nothing is lost if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/relevia-research

In [ ]:
!pip install -q -r requirements.txt

## 2. Log in to Hugging Face
Needed for the Llama models (accept Meta's licence on the model page first). Qwen doesn't need it.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Build the dataset (once)

In [ ]:
!python data/build_dataset.py
!cat data/processed/split_info.json

## 4. Pick a model

In [ ]:
MODEL = 'configs/models/llama-3.2-3b.yaml'   # or llama-3.2-1b / qwen2.5-3b / qwen2.5-1.5b
SEED = 1

## 5. Smoke test first (a few minutes)
If these fail, fix them before spending hours on a real run.

In [ ]:
!python src/generate.py --model {MODEL} --condition zero_shot --testset val --limit 16
!python src/train.py --model {MODEL} --seed 99 --limit 200

## 6. Real runs

In [ ]:
# baselines
for cond in ['zero_shot', 'tuned_prompt']:
    for ts in ['test_seen', 'test_unseen', 'safety/crisis', 'safety/sycophancy']:
        !python src/generate.py --model {MODEL} --condition {cond} --testset {ts}

In [ ]:
# fine-tune + generate (repeat with SEED = 2 and 3)
!python src/train.py --model {MODEL} --seed {SEED}
for ts in ['test_seen', 'test_unseen', 'safety/crisis', 'safety/sycophancy']:
    !python src/generate.py --model {MODEL} --condition finetuned --testset {ts} --seed {SEED}

## 7. Metrics

In [ ]:
!python src/evaluate.py
import pandas as pd
pd.read_csv('results/metrics/auto_metrics.csv')